# Chapter 8 — Memory and Local Models

Companion notebook for **Chapter 8** of *Build an Advanced RAG Application (From Scratch)*.

The Chapter 7 pipeline treats every query as a fresh start. This chapter fixes that with two memory layers, and introduces a local model stack (Ollama) for steps that must not call a third-party API.

| Component | What it does | Module |
|-----------|--------------|--------|
| **Short-term memory** | Sliding window of recent turns + semantic retrieval over older turns | [conversation_memory.py](conversation_memory.py) |
| **Long-term memory** | Cross-session distilled facts per user, TTL-gated, deduped by category | [conversation_memory.py](conversation_memory.py) |
| **Fact extractor** | LLM-backed extractor that decides what in each turn is worth keeping | [conversation_memory.py](conversation_memory.py) |
| **Local LLM (Ollama)** | Drop-in OpenAI-compatible wrapper for fully local inference | [local_llm.py](local_llm.py) |
| **Memory-augmented pipeline** | Chapter 7 pipeline extended with both memory layers | [pipeline_ch8.py](pipeline_ch8.py) |

## 0. Setup

Required keys in your `.env` (at the repo root):

| Key | Required? | Used for |
|-----|-----------|----------|
| `OPENAI_API_KEY` | yes | Router, rewriter, decomposer, synthesis, fact extractor |
| `OLLAMA_BASE_URL` | optional | Local Ollama endpoint (default: `http://localhost:11434/v1`) |
| `OLLAMA_MODEL` | optional | Local model name (default: `llama3.2`) |
| `QDRANT_URL` | optional | Remote Qdrant. Falls back to in-memory |
| `SERPAPI_KEY` | optional | Real web search. Without it, returns a stub |

**Local model setup (optional but recommended):**
```bash
brew install ollama
ollama pull llama3.2
ollama serve
```

In [ ]:
import sys, asyncio
sys.path.insert(0, '.')
sys.path.insert(0, '../chapter_07_enterprise_rag')

import nest_asyncio
nest_asyncio.apply()

from agentic_router import get_qdrant_async_client
from semantic_cache import SemanticCaching
from ingest import ingest_all

from conversation_memory import ConversationMemory, LongTermMemory, MemoryFactExtractor
from local_llm import is_ollama_available, local_chat
from pipeline_ch8 import memory_rag_pipeline

print('Ollama available:', is_ollama_available())

## 1. Ingest corpora (same as Chapter 7)

Ingests the OpenAI agents guide and Uber/Lyft 10-Ks into an in-memory Qdrant client.
Takes ~2 minutes on CPU. Set `QDRANT_URL` to persist across sessions.

In [ ]:
qdrant = get_qdrant_async_client()
asyncio.run(ingest_all(qdrant))

## 2. Short-term memory

`ConversationMemory` keeps the last N turns verbatim (the *recent window*) and embeds
turns that age out of the window so they can be retrieved semantically.

Two retrieval modes:
- `get_recent(n)` — the last n turns, always included in the rewriter context
- `get_relevant(query)` — older turns with high embedding similarity to the new query

The key invariant: recent turns are never duplicated in the relevant set.

In [ ]:
mem = ConversationMemory(window_size=3)

# Simulate a multi-turn conversation
turns_data = [
    ("What was Uber's revenue in 2021?",
     "Uber's total revenue in 2021 was $17.5 billion, up 57% year-over-year."),
    ("How did Lyft compare?",
     "Lyft's 2021 revenue was $3.2 billion, a 36% increase from 2020."),
    ("What drove Uber's growth?",
     "Delivery (Uber Eats) was the primary driver, growing 72% to $8.3B."),
    ("What about their operating losses?",
     "Uber's operating loss was $3.8B in 2021; Lyft's was $1.0B."),
    ("Any profitability milestones?",
     "Uber achieved its first-ever adjusted EBITDA profit in Q4 2021."),
]

turns = [mem.add_turn(u, a) for u, a in turns_data]
print(f'Total turns: {len(mem)}')
print(f'Recent window ({mem._window_size}):',
      [t.user_msg[:40] for t in mem.get_recent()])

In [ ]:
# Semantic retrieval over turns that have aged out of the window
query = "Uber revenue breakdown"
relevant = mem.get_relevant(query)
print(f'Relevant older turns for "{query}":')
for t in relevant:
    print(f'  "{t.user_msg[:60]}"')

In [ ]:
# Full context string passed to the rewriter
print(mem.get_context_for_rewriter("What about their margins?"))

## 3. Local LLM (Ollama)

Some pipeline steps should not call a remote API:
- **Fact extraction** — we're processing the user's own conversation
- **Input guardrail** (Chapter 9) — we're inspecting untrusted content

`local_chat()` wraps Ollama's OpenAI-compatible endpoint. If Ollama isn't running,
the fact extractor automatically falls back to `gpt-4o-mini`.

In [ ]:
if is_ollama_available():
    response = local_chat(
        "In one sentence, what does RAG stand for and why does it matter?",
        max_tokens=80,
    )
    print('Ollama response:', response)
else:
    print('Ollama not running — start with: ollama serve')
    print('Fact extractor will fall back to gpt-4o-mini automatically.')

## 4. Long-term memory

`LongTermMemory` persists facts about the user across sessions. The `MemoryFactExtractor`
runs after each turn and decides (via LLM) whether anything durable was revealed.

Design rules:
- Store distillations, not transcripts
- Every fact has a TTL (role=180d, preference=90d, interest=30d, project=14d)
- Newer facts in the same category supersede older ones

In [ ]:
lt_mem = LongTermMemory()

# Simulate a turn where the user reveals role context
from conversation_memory import Turn
from datetime import datetime
from uuid import uuid4

reveal_turn = Turn(
    turn_id=str(uuid4()),
    user_msg="I'm on the equity research team so I mostly care about Lyft vs Uber profitability.",
    assistant_msg="Understood. For equity research, the key metrics are adjusted EBITDA margins...",
    timestamp=datetime.utcnow(),
)

facts = lt_mem.extract_and_store("user_42", reveal_turn)
print(f'Extracted {len(facts)} fact(s):')
for f in facts:
    print(f'  [{f.category}] {f.content}  (expires: {f.expires_at.date()})')

In [ ]:
# Recall facts by semantic similarity to a new query
recalled = lt_mem.recall("user_42", "What should I focus on for my analysis?")
print('Recalled facts:')
for f in recalled:
    print(f'  [{f.category}] {f.content}')

## 5. Memory-augmented pipeline

`memory_rag_pipeline` is the Chapter 7 pipeline with two additions:
1. Short-term context passed to the query rewriter
2. Long-term user facts prepended to the synthesis context

Every turn updates both memory layers automatically.

In [ ]:
cache = SemanticCaching(clear_on_init=True)
st_mem = ConversationMemory(window_size=6)
lt_mem = LongTermMemory()

conversation = [
    "What was Uber's revenue in 2021?",
    "How does that compare to Lyft?",
    "What drove their respective growth?",
    "Which company had better unit economics?",
]

for q in conversation:
    print('=' * 70)
    print(f'User: {q}')
    result = asyncio.run(
        memory_rag_pipeline(q, cache, qdrant, st_mem, lt_mem, user_id="user_42")
    )
    print(f'Rewritten: {result["rewritten_query"]}')
    print(f'Memory facts used: {result["memory_facts"]}')
    print(f'Answer: {(result["answer"] or "")[:300]}...')
    print()

## 6. What short-term memory does for vague follow-ups

Without memory, "How does that compare?" is unanswerable — the rewriter has no context.
With memory, the rewriter sees the prior exchange and produces a precise query.

In [ ]:
from query_rewriter import rewrite_query

# Without memory
print('Without memory context:')
print(' ', rewrite_query("How does that compare?"))

# With memory context
print('\nWith memory context:')
context = st_mem.get_context_for_rewriter("How does that compare?")
history = [("context", context)] if context else None
print(' ', rewrite_query("How does that compare?", conversation_history=history))

## What's next

Chapter 9 adds the final production layer on top of this pipeline:
- **Input guardrail** — blocks prompt injection and PII requests before they touch the system (runs locally via Ollama)
- **Output guardrail** — checks answers for hallucination and policy violations
- **Access control** — role-based collection permissions with provenance tracking
- **Full production pipeline** — everything wired together